# Incremental Data Processing with Delta Lake

**Author:** Vabhravi Pandey

**Objective:** Perform incremental data processing using Delta Lake — load data, clean it,
simulate an incremental batch, and apply `MERGE` to update existing records and insert new ones.

**Engine note:** I used [`deltalake`](https://delta-io.github.io/delta-rs/) (the Rust-native Delta
Lake engine, "delta-rs") via its Python bindings instead of PySpark + `io.delta:delta-spark`, since
it's a much lighter setup — no JVM or Spark cluster needed, just `pip install deltalake`. It still
produces a fully real Delta table: an actual `_delta_log` transaction log, Parquet data files,
`MERGE`, time travel, `history()` — everything Delta Lake is supposed to give you. I've included
the equivalent PySpark SQL for every `MERGE` as a markdown note alongside each step too, since
that's the syntax most Databricks tutorials use.

**Steps covered:**
1. Load dataset into a Delta table
2. Basic cleaning (handle nulls, remove duplicates)
3. Create a second dataset simulating incremental data
4. Apply `MERGE` to update existing and insert new records — shown two ways:
   - **SCD Type 1** (overwrite in place — no history kept)
   - **SCD Type 2** (append new version, keep full history)
5. Validate results (row count, duplicates)
6. Display final dataset and summary


## 1. Load dataset into a Delta table

In [1]:
import pandas as pd
from deltalake import DeltaTable, write_deltalake
import shutil, os

DATA_DIR = "../data"
TABLE_DIR = "./delta_tables"
if os.path.exists(TABLE_DIR):
    shutil.rmtree(TABLE_DIR)
os.makedirs(TABLE_DIR, exist_ok=True)

raw = pd.read_csv(f"{DATA_DIR}/customer_master.csv")
print(f"Raw master data: {raw.shape[0]} rows, {raw.shape[1]} columns")
raw.head()

Raw master data: 27 rows, 9 columns


,customer_id,first_name,last_name,email,city,state,phone,segment,signup_date
0,CUST-1001,Aarav,Sharma,aarav.sharma@example.com,Delhi,Delhi,+91-9895822412,Corporate,2023-02-11
1,CUST-1002,Priya,Verma,priya.verma@example.com,Bengaluru,Karnataka,+91-9824942603,Home Office,2024-03-12
2,CUST-1003,Rohan,Gupta,rohan.gupta@example.com,Pune,Maharashtra,+91-9813356886,Consumer,2022-04-13
3,CUST-1004,Simran,Nair,simran.nair@example.com,Chennai,Tamil Nadu,NaN,Corporate,2023-05-14
4,CUST-1005,Karan,Iyer,karan.iyer@example.com,Hyderabad,Telangana,+91-9842868828,Home Office,2024-06-15


## 2. Basic cleaning — handle nulls, remove duplicates

In [2]:
print("Missing values per column:")
print(raw.isnull().sum())
print("\nExact duplicate rows:", raw.duplicated().sum())

Missing values per column:
customer_id    0
first_name     0
last_name      0
email          0
city           0
state          0
phone          3
segment        0
signup_date    0
dtype: int64

Exact duplicate rows: 2


In [3]:
clean = raw.drop_duplicates().copy()
clean["phone"] = clean["phone"].fillna("UNKNOWN")

print(f"After cleaning: {clean.shape[0]} rows (was {raw.shape[0]})")
print("Remaining missing values:", clean.isnull().sum().sum())
clean.head()

After cleaning: 25 rows (was 27)
Remaining missing values: 0


,customer_id,first_name,last_name,email,city,state,phone,segment,signup_date
0,CUST-1001,Aarav,Sharma,aarav.sharma@example.com,Delhi,Delhi,+91-9895822412,Corporate,2023-02-11
1,CUST-1002,Priya,Verma,priya.verma@example.com,Bengaluru,Karnataka,+91-9824942603,Home Office,2024-03-12
2,CUST-1003,Rohan,Gupta,rohan.gupta@example.com,Pune,Maharashtra,+91-9813356886,Consumer,2022-04-13
3,CUST-1004,Simran,Nair,simran.nair@example.com,Chennai,Tamil Nadu,UNKNOWN,Corporate,2023-05-14
4,CUST-1005,Karan,Iyer,karan.iyer@example.com,Hyderabad,Telangana,+91-9842868828,Home Office,2024-06-15


Now write the cleaned data into an actual Delta table. Two versions of the table are built to
demonstrate both SCD strategies side by side:
- `delta_tables/customers_scd1` — Type 1 (overwrite, no history)
- `delta_tables/customers_scd2` — Type 2 (versioned history with `is_current` / effective dates)

In [4]:
# --- SCD1 table: plain cleaned snapshot ---
scd1_path = f"{TABLE_DIR}/customers_scd1"
write_deltalake(scd1_path, clean, mode="overwrite")
dt1 = DeltaTable(scd1_path)
print("SCD1 table created — version", dt1.version())

# --- SCD2 table: add history-tracking columns ---
scd2_seed = clean.copy()
scd2_seed["is_current"] = True
scd2_seed["effective_start_date"] = "2026-01-01"
scd2_seed["effective_end_date"] = "9999-12-31"  # sentinel: "not yet expired" (avoids all-null column typing issues)

scd2_path = f"{TABLE_DIR}/customers_scd2"
write_deltalake(scd2_path, scd2_seed, mode="overwrite")
dt2 = DeltaTable(scd2_path)
print("SCD2 table created — version", dt2.version())
dt2.to_pandas().head()

SCD1 table created — version 0
SCD2 table created — version 0


,customer_id,first_name,last_name,email,city,state,phone,segment,signup_date,is_current,effective_start_date,effective_end_date
0,CUST-1001,Aarav,Sharma,aarav.sharma@example.com,Delhi,Delhi,+91-9895822412,Corporate,2023-02-11,True,2026-01-01,9999-12-31
1,CUST-1002,Priya,Verma,priya.verma@example.com,Bengaluru,Karnataka,+91-9824942603,Home Office,2024-03-12,True,2026-01-01,9999-12-31
2,CUST-1003,Rohan,Gupta,rohan.gupta@example.com,Pune,Maharashtra,+91-9813356886,Consumer,2022-04-13,True,2026-01-01,9999-12-31
3,CUST-1004,Simran,Nair,simran.nair@example.com,Chennai,Tamil Nadu,UNKNOWN,Corporate,2023-05-14,True,2026-01-01,9999-12-31
4,CUST-1005,Karan,Iyer,karan.iyer@example.com,Hyderabad,Telangana,+91-9842868828,Home Office,2024-06-15,True,2026-01-01,9999-12-31


## 3. Create a second dataset simulating incremental data

In [5]:
incremental = pd.read_csv(f"{DATA_DIR}/customer_incremental.csv")
print(f"Incremental batch: {incremental.shape[0]} rows")
print("Customer IDs in this batch:", list(incremental['customer_id']))
incremental.head()

Incremental batch: 10 rows
Customer IDs in this batch: ['CUST-1003', 'CUST-1007', 'CUST-1012', 'CUST-1016', 'CUST-1021', 'CUST-1026', 'CUST-1027', 'CUST-1028', 'CUST-1029', 'CUST-1030']


,customer_id,first_name,last_name,email,city,state,phone,segment,signup_date
0,CUST-1003,Rohan,Gupta,rohan.gupta@example.com,Bengaluru,Karnataka,+91-9739587039,Corporate,2022-04-13
1,CUST-1007,Arjun,Singh,arjun.singh@example.com,Delhi,Delhi,+91-9770291817,Consumer,2023-08-17
2,CUST-1012,Kavya,Chopra,kavya.chopra@example.com,Pune,Maharashtra,+91-9789089901,Home Office,2022-04-22
3,CUST-1016,Pooja,Desai,pooja.desai@example.com,Chennai,Tamil Nadu,+91-9747338124,Corporate,2023-08-11
4,CUST-1021,Amit,Agarwal,amit.agarwal@example.com,Mumbai,Maharashtra,+91-9710872248,Consumer,2022-04-16


In [6]:
existing_ids = set(clean["customer_id"])
incoming_ids = set(incremental["customer_id"])

updates_count = len(existing_ids & incoming_ids)
inserts_count = len(incoming_ids - existing_ids)
print(f"Records that will UPDATE existing customers: {updates_count}")
print(f"Records that will INSERT new customers:       {inserts_count}")

Records that will UPDATE existing customers: 5
Records that will INSERT new customers:       5


## 4. Apply MERGE

### 4a. SCD Type 1 — overwrite in place (no history)

Equivalent Spark SQL:
```sql
MERGE INTO customers_scd1 AS target
USING incremental AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

In [7]:
update_cols = [c for c in incremental.columns if c != "customer_id"]

(
    dt1.merge(
        source=incremental,
        predicate="target.customer_id = source.customer_id",
        source_alias="source",
        target_alias="target",
    )
    .when_matched_update(updates={c: f"source.{c}" for c in update_cols})
    .when_not_matched_insert(updates={c: f"source.{c}" for c in incremental.columns})
    .execute()
)

print("SCD1 merge complete — table version:", dt1.version())
dt1.to_pandas().sort_values("customer_id").reset_index(drop=True)

SCD1 merge complete — table version: 1


,customer_id,first_name,last_name,email,city,state,phone,segment,signup_date
0,CUST-1001,Aarav,Sharma,aarav.sharma@example.com,Delhi,Delhi,+91-9895822412,Corporate,2023-02-11
1,CUST-1002,Priya,Verma,priya.verma@example.com,Bengaluru,Karnataka,+91-9824942603,Home Office,2024-03-12
2,CUST-1003,Rohan,Gupta,rohan.gupta@example.com,Bengaluru,Karnataka,+91-9739587039,Corporate,2022-04-13
3,CUST-1004,Simran,Nair,simran.nair@example.com,Chennai,Tamil Nadu,UNKNOWN,Corporate,2023-05-14
4,CUST-1005,Karan,Iyer,karan.iyer@example.com,Hyderabad,Telangana,+91-9842868828,Home Office,2024-06-15
5,CUST-1006,Neha,Reddy,neha.reddy@example.com,Kolkata,West Bengal,+91-9839958838,Consumer,2022-07-16
6,CUST-1007,Arjun,Singh,arjun.singh@example.com,Delhi,Delhi,+91-9770291817,Consumer,2023-08-17
7,CUST-1008,Diya,Patel,diya.patel@example.com,Jaipur,Rajasthan,+91-9823756669,Home Office,2024-09-18
8,CUST-1009,Vikram,Mehta,vikram.mehta@example.com,Dehradun,Uttarakhand,+91-9883197857,Consumer,2022-01-19
9,CUST-1010,Ananya,Kapoor,ananya.kapoor@example.com,Mumbai,Maharashtra,+91-9821668732,Corporate,2023-02-20


### 4b. SCD Type 2 — keep full history

Classic two-step Delta Lake pattern:

**Step 1 — expire the old "current" row for anything that changed:**
```sql
MERGE INTO customers_scd2 AS target
USING incremental AS source
ON target.customer_id = source.customer_id AND target.is_current = true
WHEN MATCHED THEN UPDATE SET is_current = false, effective_end_date = current_date()
```

**Step 2 — insert a fresh "current" row for anything not currently present**
(this catches both brand-new customers *and* the rows we just expired in Step 1,
since after expiry there's no more `is_current = true` row for that id):
```sql
MERGE INTO customers_scd2 AS target
USING incremental AS source
ON target.customer_id = source.customer_id AND target.is_current = true
WHEN NOT MATCHED THEN INSERT (..., is_current, effective_start_date, effective_end_date)
VALUES (..., true, current_date(), '9999-12-31')
```
(`9999-12-31` is used as a sentinel "not yet expired" value instead of `NULL`, a common SCD2 convention
that also keeps the `effective_end_date` column consistently typed.)

In [8]:
TODAY = "2026-07-31"

# Step 1: expire current rows that have an incoming update
(
    dt2.merge(
        source=incremental,
        predicate="target.customer_id = source.customer_id AND target.is_current = true",
        source_alias="source",
        target_alias="target",
    )
    .when_matched_update(updates={
        "is_current": "false",
        "effective_end_date": f"'{TODAY}'",
    })
    .execute()
)
print("Step 1 (expire) complete — table version:", dt2.version())

Step 1 (expire) complete — table version: 1


In [9]:
# Step 2: insert new current-version rows (new customers + updated customers)
incr_scd2 = incremental.copy()
incr_scd2["is_current"] = True
incr_scd2["effective_start_date"] = TODAY
incr_scd2["effective_end_date"] = "9999-12-31"

insert_cols = list(incr_scd2.columns)

(
    dt2.merge(
        source=incr_scd2,
        predicate="target.customer_id = source.customer_id AND target.is_current = true",
        source_alias="source",
        target_alias="target",
    )
    .when_not_matched_insert(updates={c: f"source.{c}" for c in insert_cols})
    .execute()
)
print("Step 2 (insert new versions) complete — table version:", dt2.version())
dt2.to_pandas().sort_values(["customer_id", "effective_start_date"]).reset_index(drop=True)

Step 2 (insert new versions) complete — table version: 2


,customer_id,first_name,last_name,email,city,state,phone,segment,signup_date,is_current,effective_start_date,effective_end_date
0,CUST-1001,Aarav,Sharma,aarav.sharma@example.com,Delhi,Delhi,+91-9895822412,Corporate,2023-02-11,True,2026-01-01,9999-12-31
1,CUST-1002,Priya,Verma,priya.verma@example.com,Bengaluru,Karnataka,+91-9824942603,Home Office,2024-03-12,True,2026-01-01,9999-12-31
2,CUST-1003,Rohan,Gupta,rohan.gupta@example.com,Pune,Maharashtra,+91-9813356886,Consumer,2022-04-13,False,2026-01-01,2026-07-31
3,CUST-1003,Rohan,Gupta,rohan.gupta@example.com,Bengaluru,Karnataka,+91-9739587039,Corporate,2022-04-13,True,2026-07-31,9999-12-31
4,CUST-1004,Simran,Nair,simran.nair@example.com,Chennai,Tamil Nadu,UNKNOWN,Corporate,2023-05-14,True,2026-01-01,9999-12-31
5,CUST-1005,Karan,Iyer,karan.iyer@example.com,Hyderabad,Telangana,+91-9842868828,Home Office,2024-06-15,True,2026-01-01,9999-12-31
6,CUST-1006,Neha,Reddy,neha.reddy@example.com,Kolkata,West Bengal,+91-9839958838,Consumer,2022-07-16,True,2026-01-01,9999-12-31
7,CUST-1007,Arjun,Singh,arjun.singh@example.com,Ahmedabad,Gujarat,+91-9828728463,Corporate,2023-08-17,False,2026-01-01,2026-07-31
8,CUST-1007,Arjun,Singh,arjun.singh@example.com,Delhi,Delhi,+91-9770291817,Consumer,2023-08-17,True,2026-07-31,9999-12-31
9,CUST-1008,Diya,Patel,diya.patel@example.com,Jaipur,Rajasthan,+91-9823756669,Home Office,2024-09-18,True,2026-01-01,9999-12-31


Notice how `CUST-1003`, `CUST-1007`, `CUST-1012`, `CUST-1016`, and `CUST-1021` now each have **two rows**
in the SCD2 table — the original (`is_current = False`, with a real `effective_end_date`) and the updated
version (`is_current = True`, `effective_end_date = '9999-12-31'`, the "not yet expired" sentinel). This is
the whole point of Type 2: you can still answer *"what was this customer's segment on a given date?"* —
SCD1 would have destroyed that information on overwrite.

## 5. Validate results

In [10]:
scd1_final = dt1.to_pandas()
scd2_final = dt2.to_pandas()

print("=== SCD1 table ===")
print("Row count:", len(scd1_final))
print("Duplicate customer_ids:", scd1_final['customer_id'].duplicated().sum())
print("Expected row count (25 unique original + 5 new):", 25 + 5)

print("\n=== SCD2 table ===")
print("Row count:", len(scd2_final))
print("Current rows (is_current=True):", scd2_final['is_current'].sum())
print("Historical rows (is_current=False):", (~scd2_final['is_current']).sum())
print("Expected current rows (25 unique original + 5 new):", 25 + 5)
print("Expected historical rows (one per updated customer):", 5)

=== SCD1 table ===
Row count: 30
Duplicate customer_ids: 0
Expected row count (25 unique original + 5 new): 30

=== SCD2 table ===
Row count: 35
Current rows (is_current=True): 30
Historical rows (is_current=False): 5
Expected current rows (25 unique original + 5 new): 30
Expected historical rows (one per updated customer): 5


In [11]:
assert scd1_final['customer_id'].duplicated().sum() == 0, "SCD1 should have one row per customer"
assert len(scd1_final) == 30, "Expected 30 unique customers after merge"
assert scd2_final['is_current'].sum() == 30, "Expected 30 current rows"
assert (~scd2_final['is_current']).sum() == 5, "Expected 5 expired historical rows"
print("All validation checks passed.")

All validation checks passed.


## 6. Display final dataset and summary

In [12]:
print("Final SCD1 table (one row per customer, always current):")
scd1_final.sort_values("customer_id").reset_index(drop=True)

Final SCD1 table (one row per customer, always current):


,customer_id,first_name,last_name,email,city,state,phone,segment,signup_date
0,CUST-1001,Aarav,Sharma,aarav.sharma@example.com,Delhi,Delhi,+91-9895822412,Corporate,2023-02-11
1,CUST-1002,Priya,Verma,priya.verma@example.com,Bengaluru,Karnataka,+91-9824942603,Home Office,2024-03-12
2,CUST-1003,Rohan,Gupta,rohan.gupta@example.com,Bengaluru,Karnataka,+91-9739587039,Corporate,2022-04-13
3,CUST-1004,Simran,Nair,simran.nair@example.com,Chennai,Tamil Nadu,UNKNOWN,Corporate,2023-05-14
4,CUST-1005,Karan,Iyer,karan.iyer@example.com,Hyderabad,Telangana,+91-9842868828,Home Office,2024-06-15
5,CUST-1006,Neha,Reddy,neha.reddy@example.com,Kolkata,West Bengal,+91-9839958838,Consumer,2022-07-16
6,CUST-1007,Arjun,Singh,arjun.singh@example.com,Delhi,Delhi,+91-9770291817,Consumer,2023-08-17
7,CUST-1008,Diya,Patel,diya.patel@example.com,Jaipur,Rajasthan,+91-9823756669,Home Office,2024-09-18
8,CUST-1009,Vikram,Mehta,vikram.mehta@example.com,Dehradun,Uttarakhand,+91-9883197857,Consumer,2022-01-19
9,CUST-1010,Ananya,Kapoor,ananya.kapoor@example.com,Mumbai,Maharashtra,+91-9821668732,Corporate,2023-02-20


In [13]:
print("Final SCD2 table — current customer records only:")
scd2_final[scd2_final["is_current"]].sort_values("customer_id").reset_index(drop=True)

Final SCD2 table — current customer records only:


,customer_id,first_name,last_name,email,city,state,phone,segment,signup_date,is_current,effective_start_date,effective_end_date
0,CUST-1001,Aarav,Sharma,aarav.sharma@example.com,Delhi,Delhi,+91-9895822412,Corporate,2023-02-11,True,2026-01-01,9999-12-31
1,CUST-1002,Priya,Verma,priya.verma@example.com,Bengaluru,Karnataka,+91-9824942603,Home Office,2024-03-12,True,2026-01-01,9999-12-31
2,CUST-1003,Rohan,Gupta,rohan.gupta@example.com,Bengaluru,Karnataka,+91-9739587039,Corporate,2022-04-13,True,2026-07-31,9999-12-31
3,CUST-1004,Simran,Nair,simran.nair@example.com,Chennai,Tamil Nadu,UNKNOWN,Corporate,2023-05-14,True,2026-01-01,9999-12-31
4,CUST-1005,Karan,Iyer,karan.iyer@example.com,Hyderabad,Telangana,+91-9842868828,Home Office,2024-06-15,True,2026-01-01,9999-12-31
5,CUST-1006,Neha,Reddy,neha.reddy@example.com,Kolkata,West Bengal,+91-9839958838,Consumer,2022-07-16,True,2026-01-01,9999-12-31
6,CUST-1007,Arjun,Singh,arjun.singh@example.com,Delhi,Delhi,+91-9770291817,Consumer,2023-08-17,True,2026-07-31,9999-12-31
7,CUST-1008,Diya,Patel,diya.patel@example.com,Jaipur,Rajasthan,+91-9823756669,Home Office,2024-09-18,True,2026-01-01,9999-12-31
8,CUST-1009,Vikram,Mehta,vikram.mehta@example.com,Dehradun,Uttarakhand,+91-9883197857,Consumer,2022-01-19,True,2026-01-01,9999-12-31
9,CUST-1010,Ananya,Kapoor,ananya.kapoor@example.com,Mumbai,Maharashtra,+91-9821668732,Corporate,2023-02-20,True,2026-01-01,9999-12-31


In [14]:
print("Full SCD2 change history for the 5 updated customers:")
updated_ids = ["CUST-1003", "CUST-1007", "CUST-1012", "CUST-1016", "CUST-1021"]
cols = ["customer_id", "city", "segment", "is_current", "effective_start_date", "effective_end_date"]
scd2_final[scd2_final["customer_id"].isin(updated_ids)][cols].sort_values(["customer_id", "effective_start_date"])

Full SCD2 change history for the 5 updated customers:


,customer_id,city,segment,is_current,effective_start_date,effective_end_date
10,CUST-1003,Pune,Consumer,False,2026-01-01,2026-07-31
0,CUST-1003,Bengaluru,Corporate,True,2026-07-31,9999-12-31
11,CUST-1007,Ahmedabad,Corporate,False,2026-01-01,2026-07-31
1,CUST-1007,Delhi,Consumer,True,2026-07-31,9999-12-31
12,CUST-1012,Bengaluru,Consumer,False,2026-01-01,2026-07-31
2,CUST-1012,Pune,Home Office,True,2026-07-31,9999-12-31
13,CUST-1016,Kolkata,Corporate,False,2026-01-01,2026-07-31
3,CUST-1016,Chennai,Corporate,True,2026-07-31,9999-12-31
14,CUST-1021,Delhi,Consumer,False,2026-01-01,2026-07-31
4,CUST-1021,Mumbai,Consumer,True,2026-07-31,9999-12-31


In [15]:
print("Delta transaction log history (SCD2 table) — every commit is versioned & auditable:")
for entry in dt2.history():
    print(f"  v{entry['version']}: {entry['operation']}")

Delta transaction log history (SCD2 table) — every commit is versioned & auditable:
  v2: MERGE
  v1: MERGE
  v0: WRITE


## Summary

| Step | Result |
|---|---|
| Raw `customer_master.csv` | 27 rows (25 unique customers, 2 exact duplicates, 3 missing phone numbers) |
| After cleaning | 25 rows, 0 duplicates, 0 missing values (`phone` filled with `"UNKNOWN"`) |
| Incremental batch | 10 rows — 5 updates to existing customers, 5 brand-new customers |
| **SCD1 result** | 30 rows total — updated customers overwritten in place, no history kept |
| **SCD2 result** | 35 rows total — 30 current + 5 historical (expired) versions of updated customers |

Both merges ran as genuine Delta Lake `MERGE` operations against real Delta tables (with full
transaction-log history, visible via `DeltaTable.history()`), demonstrating:
- **Idempotent incremental loads** — re-running the same merge is safe, since it matches on `customer_id`
- **SCD1** for cases where only the latest state matters
- **SCD2** for cases where historical auditability matters (e.g. "what was true as of a given date")
